# 정규화 파일 → Aiven MySQL 적재

실제 CSV/Excel 컬럼명을 그대로 사용합니다. 게시글은 `<플랫폼>_post`, 키워드는 `<플랫폼>_keyword`, 게시글-키워드는 `<플랫폼>_post_keyword` 규칙으로 통일합니다. 날짜·시간은 이미 한국 시간 기준이므로 파싱·시간대 변환 없이 원본 문자열 그대로 저장합니다.

당근에는 이름에 `최종`이 포함된 CSV가 없으므로 실제 최종 `.xlsx` 2개를 사용합니다.

In [ ]:
# 최초 1회만 실행하세요.
# %pip install -q pandas sqlalchemy pymysql cryptography openpyxl python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Aiven 접속 정보

Aiven Console의 접속 정보를 `.env`에 설정하세요.

```text
MYSQL_HOST=서비스명.aivencloud.com
MYSQL_PORT=12345
MYSQL_USER=integral_ej
MYSQL_PASSWORD=비밀번호
MYSQL_DATABASE=daewon
```

In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / '02_csv').is_dir() and (candidate / '03_code').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
CSV_ROOT = PROJECT_ROOT / '02_csv'
ENV_PATH = PROJECT_ROOT / '03_code' / '01_python' / '05_merge' / '.env'
load_dotenv(ENV_PATH)
DB = {
    'host': os.getenv('MYSQL_HOST'),
    'port': int(os.getenv('MYSQL_PORT', '3306')),
    'user': os.getenv('MYSQL_USER', 'integral_ej'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'database': os.getenv('MYSQL_DATABASE', 'daewon'),
}
if not DB['host'] or not DB['password']:
    raise RuntimeError(f'{ENV_PATH}에 MYSQL_HOST와 MYSQL_PASSWORD를 설정하세요.')

db_url = URL.create(
    'mysql+pymysql', username=DB['user'], password=DB['password'],
    host=DB['host'], port=DB['port'], database=DB['database'],
    query={'charset': 'utf8mb4'}
)
engine = create_engine(db_url, pool_pre_ping=True, pool_recycle=300, future=True)
with engine.connect() as conn:
    info = conn.execute(text('SELECT DATABASE(), VERSION()')).one()
print('Aiven MySQL 연결 성공:', info)

Aiven MySQL 연결 성공: ('daewon', '8.4.11')


## 2. 파일 선택 및 원본 컬럼 확인

In [2]:
def exactly_one(paths, label):
    paths = sorted(paths)
    if len(paths) != 1:
        raise FileNotFoundError(f'{label}: 파일이 1개여야 하지만 {len(paths)}개입니다: {paths}')
    return paths[0]

instagram = CSV_ROOT / '02_instagram'
x_dir = CSV_ROOT / '03_x' / 'normalized'
daangn = CSV_ROOT / '04_daangn'
FILES = {
    'instagram_keyword': exactly_one(instagram.glob('*3nf_keywords.csv'), 'Instagram 키워드'),
    'instagram_media_type': exactly_one(instagram.glob('*3nf_media_types.csv'), 'Instagram 미디어 유형'),
    'instagram_post': exactly_one(instagram.glob('*3nf_posts.csv'), 'Instagram 게시글'),
    'instagram_post_keyword': exactly_one(instagram.glob('*3nf_post_keywords.csv'), 'Instagram 게시글-키워드'),
    'x_post': exactly_one(x_dir.glob('x_posts.csv'), 'X 게시글'),
    'x_source_file': exactly_one(x_dir.glob('x_source_files.csv'), 'X 원본 파일'),
    'x_post_source': exactly_one(x_dir.glob('x_post_sources.csv'), 'X 게시글 출처'),
    'x_keyword': exactly_one(x_dir.glob('x_keywords.csv'), 'X 키워드'),
    'x_post_keyword': exactly_one(x_dir.glob('x_post_keywords.csv'), 'X 게시글-키워드'),
    'daangn_post': exactly_one(daangn.glob('daangn_analysis_post_table*최종*.xlsx'), '당근 게시글'),
    'daangn_product_category': exactly_one(daangn.glob('daangn_product_category_table*최종*.xlsx'), '당근 상품 카테고리'),
}

def read_source(path):
    if path.suffix.lower() == '.csv':
        # 모든 값을 문자열로 읽어 날짜·시간과 문자열 'null'을 변환하지 않습니다.
        return pd.read_csv(path, dtype='string', encoding='utf-8-sig', keep_default_na=False, na_filter=False, on_bad_lines='error')
    # parse_dates를 사용하지 않습니다.
    return pd.read_excel(path, dtype='string', engine='openpyxl')

frames = {table: read_source(path) for table, path in FILES.items()}
display(pd.DataFrame([
    {'table': table, 'rows': len(df), 'columns': ', '.join(df.columns), 'file': str(FILES[table].relative_to(PROJECT_ROOT))}
    for table, df in frames.items()
]))

,table,rows,columns,file
0,instagram_keyword,13,"keyword_id, keyword",02_csv\02_instagram\instagram_normalized_3nf_k...
1,instagram_media_type,3,"media_type_id, media_type",02_csv\02_instagram\instagram_normalized_3nf_m...
2,instagram_post,13464,"post_id, description, media_type_id, post_url,...",02_csv\02_instagram\instagram_normalized_3nf_p...
3,instagram_post_keyword,13464,"post_id, keyword_id",02_csv\02_instagram\instagram_normalized_3nf_p...
4,x_post,5000,"post_id, post_url, account_id, account_name, t...",02_csv\03_x\normalized\x_posts.csv
5,x_source_file,5,"source_file, source_year",02_csv\03_x\normalized\x_source_files.csv
6,x_post_source,5000,"source_file, source_row, post_id, input_row_no",02_csv\03_x\normalized\x_post_sources.csv
7,x_keyword,13,keyword,02_csv\03_x\normalized\x_keywords.csv
8,x_post_keyword,7466,"post_id, keyword_order, keyword",02_csv\03_x\normalized\x_post_keywords.csv
9,daangn_post,3813,"post_id, platform, post_url, post_title, descr...",02_csv\04_daangn\daangn_analysis_post_table 최종...


## 3. 테이블 생성

컬럼명은 실제 파일 헤더를 따릅니다. 날짜·시간 컬럼은 원본 표현을 보존하기 위해 `VARCHAR(40)`으로 저장합니다.

In [3]:
DDL = [
'''CREATE TABLE IF NOT EXISTS instagram_keyword (keyword_id INT PRIMARY KEY, keyword VARCHAR(100) NOT NULL) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS instagram_media_type (media_type_id INT PRIMARY KEY, media_type VARCHAR(30) NOT NULL) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS instagram_post (
 post_id BIGINT PRIMARY KEY, description LONGTEXT NULL, media_type_id INT NOT NULL, post_url VARCHAR(500) NOT NULL, created_at VARCHAR(40) NOT NULL,
 like_count BIGINT NOT NULL, comment_count BIGINT NOT NULL, INDEX idx_instagram_post_media_type (media_type_id),
 CONSTRAINT fk_instagram_post_media_type FOREIGN KEY (media_type_id) REFERENCES instagram_media_type(media_type_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS instagram_post_keyword (
 post_id BIGINT NOT NULL, keyword_id INT NOT NULL, PRIMARY KEY (post_id, keyword_id), INDEX idx_ig_pk_keyword (keyword_id),
 CONSTRAINT fk_ig_pk_post FOREIGN KEY (post_id) REFERENCES instagram_post(post_id),
 CONSTRAINT fk_ig_pk_keyword FOREIGN KEY (keyword_id) REFERENCES instagram_keyword(keyword_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_post (
 post_id VARCHAR(32) PRIMARY KEY, post_url VARCHAR(500) NOT NULL, account_id VARCHAR(255) NOT NULL, account_name VARCHAR(255) NOT NULL,
 text LONGTEXT NOT NULL, created_at VARCHAR(40) NOT NULL, comment_count BIGINT NOT NULL, repost_count BIGINT NOT NULL, view_count BIGINT NULL, like_count BIGINT NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_source_file (source_file VARCHAR(255) PRIMARY KEY, source_year INT NOT NULL) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_post_source (
 source_file VARCHAR(255) NOT NULL, source_row BIGINT NOT NULL, post_id VARCHAR(32) NOT NULL, input_row_no BIGINT NOT NULL,
 PRIMARY KEY (source_file, source_row), INDEX idx_x_ps_post (post_id),
 CONSTRAINT fk_x_ps_file FOREIGN KEY (source_file) REFERENCES x_source_file(source_file),
 CONSTRAINT fk_x_ps_post FOREIGN KEY (post_id) REFERENCES x_post(post_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_keyword (keyword VARCHAR(255) PRIMARY KEY) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_post_keyword (
 post_id VARCHAR(32) NOT NULL, keyword_order BIGINT NOT NULL, keyword VARCHAR(255) NOT NULL, PRIMARY KEY (post_id, keyword_order), INDEX idx_x_pk_keyword (keyword),
 CONSTRAINT fk_x_pk_post FOREIGN KEY (post_id) REFERENCES x_post(post_id), CONSTRAINT fk_x_pk_keyword FOREIGN KEY (keyword) REFERENCES x_keyword(keyword)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS daangn_post (
 post_id VARCHAR(64) PRIMARY KEY, platform VARCHAR(30) NOT NULL, post_url VARCHAR(1000) NOT NULL, post_title TEXT NOT NULL, description LONGTEXT NULL,
 price BIGINT NULL, account_id VARCHAR(255) NULL, account_name VARCHAR(255) NULL, location VARCHAR(255) NULL, created_at VARCHAR(40) NULL,
 year INT NULL, month INT NULL, keyword VARCHAR(255) NOT NULL, product_category_id BIGINT NULL, trade_status VARCHAR(100) NULL,
 repost_count BIGINT NULL, view_count BIGINT NULL, like_count BIGINT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS daangn_product_category (
 platform VARCHAR(30) NOT NULL, post_id VARCHAR(64) NOT NULL, product_category_id BIGINT NOT NULL, category_name VARCHAR(255) NOT NULL,
 PRIMARY KEY (post_id, product_category_id), CONSTRAINT fk_dg_category_post FOREIGN KEY (post_id) REFERENCES daangn_post(post_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4'''
]
with engine.begin() as conn:
    for statement in DDL:
        conn.execute(text(statement))
print('테이블 생성 완료')

테이블 생성 완료


## 4. 실제 컬럼 및 PK/FK 검사

In [4]:
EXPECTED_COLUMNS = {
 'instagram_keyword': ['keyword_id', 'keyword'], 'instagram_media_type': ['media_type_id', 'media_type'],
 'instagram_post': ['post_id', 'description', 'media_type_id', 'post_url', 'created_at', 'like_count', 'comment_count'],
 'instagram_post_keyword': ['post_id', 'keyword_id'],
 'x_post': ['post_id', 'post_url', 'account_id', 'account_name', 'text', 'created_at', 'comment_count', 'repost_count', 'view_count', 'like_count'],
 'x_source_file': ['source_file', 'source_year'], 'x_post_source': ['source_file', 'source_row', 'post_id', 'input_row_no'],
 'x_keyword': ['keyword'], 'x_post_keyword': ['post_id', 'keyword_order', 'keyword'],
 'daangn_post': ['post_id', 'platform', 'post_url', 'post_title', 'description', 'price', 'account_id', 'account_name', 'location', 'created_at', 'year', 'month', 'keyword', 'product_category_id', 'trade_status', 'repost_count', 'view_count', 'like_count'],
 'daangn_product_category': ['platform', 'post_id', 'product_category_id', 'category_name'],
}
PKS = {
 'instagram_keyword': ['keyword_id'], 'instagram_media_type': ['media_type_id'],
 'instagram_post': ['post_id'], 'instagram_post_keyword': ['post_id', 'keyword_id'],
 'x_post': ['post_id'], 'x_source_file': ['source_file'], 'x_post_source': ['source_file', 'source_row'],
 'x_keyword': ['keyword'], 'x_post_keyword': ['post_id', 'keyword_order'], 'daangn_post': ['post_id'],
 'daangn_product_category': ['post_id', 'product_category_id'],
}
for table, expected in EXPECTED_COLUMNS.items():
    actual = list(frames[table].columns)
    if actual != expected:
        raise ValueError(f'{table} 컬럼 불일치\n기대: {expected}\n실제: {actual}')
    if frames[table][PKS[table]].isna().any().any() or frames[table].duplicated(PKS[table]).any():
        raise ValueError(f'{table}의 PK에 NULL 또는 중복이 있습니다.')

def assert_fk(child, child_col, parent, parent_col):
    child_values = set(frames[child][child_col].dropna().astype('string'))
    parent_values = set(frames[parent][parent_col].dropna().astype('string'))
    missing = child_values - parent_values
    if missing:
        raise ValueError(f'{child}.{child_col} FK 불일치 {len(missing)}개: {list(missing)[:5]}')

for relation in [
 ('instagram_post','media_type_id','instagram_media_type','media_type_id'),
 ('instagram_post_keyword','post_id','instagram_post','post_id'),
 ('instagram_post_keyword','keyword_id','instagram_keyword','keyword_id'),
 ('x_post_source','source_file','x_source_file','source_file'), ('x_post_source','post_id','x_post','post_id'),
 ('x_post_keyword','post_id','x_post','post_id'), ('x_post_keyword','keyword','x_keyword','keyword'),
 ('daangn_product_category','post_id','daangn_post','post_id'),
]:
    assert_fk(*relation)
print('실제 파일 컬럼 및 PK/FK 검사 통과')

ValueError: daangn_product_category의 PK에 NULL 또는 중복이 있습니다.

## 5. Upsert

날짜·시간을 포함한 원본 문자열은 수정하지 않습니다. nullable 컬럼의 빈 값만 SQL `NULL`로 처리합니다.

In [ ]:
NULLABLE = {
 'instagram_post': {'description'}, 'x_post': {'view_count'},
 'daangn_post': {'description','price','account_id','account_name','location','created_at','year','month','product_category_id','trade_status','repost_count','view_count','like_count'},
}
def db_value(value, nullable):
    if value is None or pd.isna(value): return None
    if isinstance(value, str): return None if nullable and value == '' else value
    return value.item() if hasattr(value, 'item') else value

def upsert(conn, table, df, pk, chunk_size=300):
    columns = list(df.columns)
    inserts = ', '.join(f'`{c}`' for c in columns)
    params = ', '.join(f':{c}' for c in columns)
    updates = [c for c in columns if c not in pk]
    update_sql = ', '.join(f'`{c}`=VALUES(`{c}`)' for c in updates)
    if not update_sql: update_sql = f'`{pk[0]}`=VALUES(`{pk[0]}`)'
    sql = text(f'INSERT INTO `{table}` ({inserts}) VALUES ({params}) ON DUPLICATE KEY UPDATE {update_sql}')
    nullable = NULLABLE.get(table, set())
    rows = [{c: db_value(v, c in nullable) for c, v in row.items()} for row in df.to_dict('records')]
    for start in range(0, len(rows), chunk_size): conn.execute(sql, rows[start:start + chunk_size])
    return len(rows)

LOAD_ORDER = ['instagram_keyword','instagram_media_type','instagram_post','instagram_post_keyword',
              'x_post','x_source_file','x_keyword','x_post_source','x_post_keyword','daangn_post','daangn_product_category']
with engine.begin() as conn:
    for table in LOAD_ORDER:
        print(f'{table}: {upsert(conn, table, frames[table], PKS[table]):,}행 처리')
print('모든 테이블 적재 완료')

## 6. 결과 확인

In [ ]:
result = []
with engine.connect() as conn:
    for table in LOAD_ORDER:
        db_rows = conn.execute(text(f'SELECT COUNT(*) FROM `{table}`')).scalar_one()
        result.append({'table': table, 'source_rows': len(frames[table]), 'database_rows': db_rows, 'ok': db_rows >= len(frames[table])})
result = pd.DataFrame(result)
display(result)
if not result['ok'].all(): raise RuntimeError('DB 행 수가 원본보다 적은 테이블이 있습니다.')